In [ ]:
!pip install requests
!pip install dotenv

ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [49]:
import requests
import json
import time
import re  # Regex for extracting Planner Plan ID from URLs
import urllib.parse
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Access token from .env
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")

In [9]:
# Your teams data (Replace with your actual teams list)
teams_data = {
  "teams": [
	{
  	"project_name": "LIDL",
  	"id": "1d092b0f-f8eb-459c-b391-f4487e66680f"
	},
	{
  	"project_name": "ACTIV GROUP",
  	"id": "d3a1f29a-40aa-4e27-b286-21f7dc5f3f7b"
	},
	{
  	"project_name": "EMPRESES Propies",
  	"id": "76a94e4e-6ae3-4b9f-a088-cc0379b876ab"
	},
	{
  	"project_name": "GESTIO RECURSOS",
  	"id": "b093c223-7d27-4e94-abf3-8c8d49122787"
	},
	{
  	"project_name": "ANIMALS SAPIENS-LLEIDA",
  	"id": "ef98578d-ba61-4a9e-844f-0a6acdcba015"
	},
	{
  	"project_name": "TENBRINKE",
  	"id": "94e913fe-d4a1-42be-86b5-dd4db1bfffa4"
	},
	{
  	"project_name": "Estudios Economicos-Rentabilidad",
  	"id": "0c106d2e-16eb-44e5-b1a2-4d8944195037"
	},
	{
  	"project_name": "ELISEO PLA 2201C",
  	"id": "19cf331d-ab77-4179-ade1-ce1def8871c6"
	},
	{
  	"project_name": "CONTABILIDAD",
  	"id": "997bdadc-64b4-4e2e-b30f-e21b461f5271"
	},
	{
  	"project_name": "ANIMALS SAPIENS",
  	"id": "b1c2eea3-e64c-4f1c-a2f2-a288f3898ad0"
	},
	{
  	"project_name": "BRM",
  	"id": "595b2599-731f-406b-8cdf-da531bd9ce19"
	},
	{
  	"project_name": "GOU",
  	"id": "10aee77c-24d6-45ff-8235-58cc14efdf4b"
	},
	{
  	"project_name": "SERVITEC GESTIO",
  	"id": "7464eb67-5cd2-4d3b-a7b5-852b2eed7a99"
	},
	{
  	"project_name": "BRICOCENTRO",
  	"id": "d316b59c-479e-4b8c-a2d3-059ba5d7b65d"
	},
	{
  	"project_name": "CELURBA",
  	"id": "333e1665-4dd4-4491-8541-d2994612ac5e"
	},
	{
  	"project_name": "IMPLANTACIONES",
  	"id": "2447a659-5dc5-405f-a5a1-7f83c0c957c3"
	},
	{
  	"project_name": "GABARRO",
  	"id": "22764f6a-f8e0-4501-9101-afb4621dee78"
	},
	{
  	"project_name": "BAUHAUS",
  	"id": "17c2772c-ed0f-4c8d-b646-61e87b5079a0"
	},
	{
  	"project_name": "JOCALBERT",
  	"id": "dc20fd8c-72ad-45d6-8c5b-c2c4f3cfd4d7"
	},
	{
  	"project_name": "PRIMAFRIO EXT",
  	"id": "a1cdf9e3-a48c-4385-93e1-39d4fb66af65"
	},
	{
  	"project_name": "STEN",
  	"id": "3e23af77-9309-45d2-8fae-66655388dca0"
	},
	{
  	"project_name": "TERADIS Hospitalet",
  	"id": "14e05916-b587-4ef8-a5a5-72555271bd45"
	},
	{
  	"project_name": "LUYFLEX 2301A",
  	"id": "0c4a1cb6-f086-479b-b19a-32c48238ff1d"
	},
	{
  	"project_name": "EUROPA CENTER 2302E",
  	"id": "15aafb33-00ac-4077-83a4-2b11657f1b66"
	},
	{
  	"project_name": "FINQUES",
  	"id": "5fd9085c-867c-4df0-8852-c2291c46dfb1"
	},
	{
  	"project_name": "VERDECORA-2404E",
  	"id": "271dda2c-b39a-4094-a69b-8a3674a5c7e0"
	},
	{
  	"project_name": "FAMILY ENERGY Gasolineras",
  	"id": "b3bf7e23-4c23-4ab3-a343-6026733ff246"
	},
	{
  	"project_name": "NATH",
  	"id": "17180815-63ab-4ff4-be78-8b8f98eccb02"
	},
	{
  	"project_name": "FAMILY CASH Hipermercats",
  	"id": "d30d38bb-4e42-45b0-b135-8ceaed995c3b"
	},
	{
  	"project_name": "FAMILY CASH Zaragoza",
  	"id": "69643937-69a8-4b69-a5e8-7d81e638a294"
	},
	{
  	"project_name": "FAMILY CASH",
  	"id": "c668a2f7-54d7-4c13-bf8e-32505e194184"
	},
	{
  	"project_name": "ACTION",
  	"id": "c81c7ff6-464f-4f88-aeea-29871082dc49"
	}
  ]
}




In [10]:
# API Headers
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json"
}


In [45]:
# Store results
channel_planners_detected = []

def extract_plan_id_from_planner_url(planner_url: str) -> str:
    """
    Extracts the 'real' Plan ID from a Microsoft Teams Planner tab URL.
    1) Decodes the overall URL (which contains a webUrl param).
    2) Finds the path segment after '/PlanViews/' if present.
    3) Otherwise looks for '?planId=' in the decoded string.
    Returns an empty string if nothing is found.
    """
    if not planner_url:
        return ""
    
    # Step 1: Decode the entire Teams URL
    decoded_url = urllib.parse.unquote(planner_url)
    
    # The real plan URL is inside the webUrl=... part. Let's safely extract it:
    # Example: ...?webUrl=<ENCODED_TASKS_URL>&label=...
    parsed = urllib.parse.urlparse(decoded_url)
    qs = urllib.parse.parse_qs(parsed.query)
    
    # webUrl might be in qs["webUrl"] if it exists
    # It's typically a list with one element if present
    if "webUrl" in qs and qs["webUrl"]:
        real_planner_link = qs["webUrl"][0]
    else:
        # If for some reason webUrl is missing, just fallback to the full decoded_url
        real_planner_link = decoded_url

    # Step 2: In that real Planner link, look for '/PlanViews/XXXXX?'
    plan_id_match = re.search(r'/PlanViews/([^?&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)

    # Step 3 (fallback): Look for planId=XXXX if the above fails
    plan_id_match = re.search(r'[?&]planId=([^&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)

    return ""  # If all else fails, return empty string


# Function to clean "tt." formatted Planner IDs (removing incorrect prefixes)
def clean_plan_id(plan_id):
    """Extracts the correct ID from messy tt. planner_id values"""
    if not plan_id:
        return None

    match = re.search(r"tt\.([a-zA-Z0-9-_]+)", plan_id)  # Extract after "tt."
    return f"tt.{match.group(1)}" if match else None  # Return cleaned Planner ID

# Function to check if a tab is a valid Planner Tab
def is_planner_tab(tab_name, planner_id, planner_url):
    """Determines if a tab is a valid Planner tab"""
    if planner_id and planner_id.startswith("tt."):  # Confirm it's a valid Planner ID
        return True
    
    if "planner" in tab_name.lower() or "tasks" in tab_name.lower():
        return True  # Likely a Planner tab

    if planner_url and "tasks.office.com" in planner_url:
        return True  # The URL confirms it's Planner

    return False  # Not a Planner tab

# Loop through each team
for team in teams_data["teams"]:
    team_id = team["id"]
    project_name = team["project_name"]

    print(f"\n🔍 Fetching channels for team: {project_name} ({team_id})")

    # Step 1: Get all channels in the team
    channels_url = f"https://graph.microsoft.com/v1.0/teams/{team_id}/channels"
    channels_response = requests.get(channels_url, headers=headers)

    if channels_response.status_code == 200:
        channels = channels_response.json().get("value", [])
    else:
        continue  # Skip this team if there's an error

    # Step 2: Check each channel for a Planner tab
    for channel in channels:
        channel_name = channel["displayName"]
        channel_id = channel["id"]
        detected_planner_tabs = []

        # Get tabs in the channel
        tabs_url = f"https://graph.microsoft.com/v1.0/teams/{team_id}/channels/{channel_id}/tabs"
        tabs_response = requests.get(tabs_url, headers=headers)

        if tabs_response.status_code == 200:
            tabs = tabs_response.json().get("value", [])

            # Loop through each tab
            for tab in tabs:
                tab_name = tab.get("displayName", "")
                planner_id = tab.get("configuration", {}).get("entityId")
                planner_url = tab.get("webUrl")

                # Extract the real Plan ID from the URL
                extracted_plan_id = extract_plan_id_from_planner_url(planner_url)

                # If a valid Planner ID exists in the URL, use that instead
                if extracted_plan_id:
                    planner_id = extracted_plan_id
                else:
                    planner_id = clean_plan_id(planner_id)  # Clean existing planner_id

                # Check if this is a valid Planner tab
                if planner_id and is_planner_tab(tab_name, planner_id, planner_url):
                    detected_planner_tabs.append({
                        "tab_name": tab_name,
                        "planner_id": planner_id,  # Now correctly extracted from URL
                        "planner_url": planner_url,
                    })

                    # Print only when a valid Planner tab is found
                    print(f"✅ Planner Tab Found: {tab_name} ({planner_id})")

        # Store detected Planner Tabs only if any were found
        if detected_planner_tabs:
            channel_planners_detected.append({
                "project_name": project_name,
                "team_id": team_id,
                "channel_name": channel_name,
                "channel_id": channel_id,
                "planner_tabs": detected_planner_tabs  # Storing only valid "tt." prefixed IDs
            })

    time.sleep(1)  # Avoid hitting API rate limits



🔍 Fetching channels for team: LIDL (1d092b0f-f8eb-459c-b391-f4487e66680f)

🔍 Fetching channels for team: ACTIV GROUP (d3a1f29a-40aa-4e27-b286-21f7dc5f3f7b)
✅ Planner Tab Found: Tasks ACTIV GROUP TERRASSA (QqtnCj433E-tc_EuxF0YHZcAGsto)
✅ Planner Tab Found: Tareas ACTIV TERRASSA (QqtnCj433E-tc_EuxF0YHZcAGsto)
✅ Planner Tab Found: Tareas ACTIV TERRASSA (1) (QqtnCj433E-tc_EuxF0YHZcAGsto)

🔍 Fetching channels for team: EMPRESES Propies (76a94e4e-6ae3-4b9f-a088-cc0379b876ab)
✅ Planner Tab Found: Tasks EMPRESES Propies (0_OQICyTFEWCPtNSsrx3RpcAG47T)

🔍 Fetching channels for team: GESTIO RECURSOS (b093c223-7d27-4e94-abf3-8c8d49122787)
✅ Planner Tab Found: TASK DELINEACIO (pKWEfYMX7U6fI_5Pht1J2pcAE5m6)

🔍 Fetching channels for team: ANIMALS SAPIENS-LLEIDA (ef98578d-ba61-4a9e-844f-0a6acdcba015)

🔍 Fetching channels for team: TENBRINKE (94e913fe-d4a1-42be-86b5-dd4db1bfffa4)

🔍 Fetching channels for team: Estudios Economicos-Rentabilidad (0c106d2e-16eb-44e5-b1a2-4d8944195037)

🔍 Fetching channels

In [46]:
# Save results to JSON file
with open("teams_data.json", "w") as file:
    json.dump(channel_planners_detected, file, indent=4)

print("\n🎯 Planner Tabs Detection Saved to teams_data.json")


🎯 Planner Tabs Detection Saved to teams_data.json
